# MB6 · Shapley, intégrales de chemin et équité — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/mb6_shapley_equite/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/mb6_shapley_equite/03_notebook.ipynb`, créée par `python tools/start_chapter.py MB6`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | MB6.1 à MB6.7, MB6.11 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | MB6.13 | Les poids des coalitions dans la valeur de Shapley, exacts ou en flottants · `coalition_weights` | 🔨 | ★ | 15 |
|  | MB6.14 | Les dividendes de Harsanyi d'un jeu donné par sa fonction caractéristique · `harsanyi_dividends` | 🔨 | ★★ | 25 |
|  | MB6.15 | L'intégrale d'un champ de vecteurs le long d'une ligne brisée, par la méthode de Simpson sur chaque segment · `line_integral` | 🔨 | ★★ | 30 |
|  | MB6.16 | La corrélation de rangs de Spearman, ex aequo compris · `spearman_rho` | 🔨 | ★ | 20 |
| B | MB6.17 | Prédire l'intégrale d'un champ de gradient et d'un champ tournant le long d'une boucle triangulaire, dans un sens, dans l'autre et deux fois, puis vérifier | 🔮 | ★ | 15 |
|  | MB6.18 | Deux mesures corrélées : la moyenne marginale (dépendance partielle) et la moyenne conditionnelle d'une même fonction, simulées pour plusieurs corrélations et comparées aux formules gaussiennes | 🔬 | ★★ | 25 |
|  | MB6.19 | Les lignes de niveau d'une fonction de deux variables, deux chemins entre les mêmes points, et l'intégrale du gradient cumulée le long de chacun | 🎨 | ★ | 25 |
| C | MB6.20 | Chasse à l'erreur : des poids de coalition sans le facteur 1/n, des ex aequo classés dans l'ordre d'apparition, une interpolation linéaire aux poids échangés | 🐛 | ★ | 20 |
|  | MB6.21 | Un test de propriété sur des lignes brisées tirées au hasard : le chemin inversé change le signe, un segment coupé en deux ne change rien, un champ de gradient donne f(fin) − f(début) | 🛠️ | ★ | 25 |
| D | MB6.22 | Défi : les indices de Shapley-Shubik exacts d'un conseil de 25 membres, sans parcourir ses coalitions une à une, en comptant par taille et par poids celles où chaque membre est pivot | 🏆 | ★★★ | 45 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB6")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : un conseil à vote pondéré, trois hameaux et une conduite d'eau, le rendement d'une parcelle, le travail de deux forces, un coût à trois facteurs, une grille de capteurs, un contrôle de pièces sur deux lignes de production, et un pont vers l'écart de deux taux de détection.
- Écrire les quatre fonctions de `mymath/attribution.py` (`coalition_weights`, `harsanyi_dividends`, `line_integral`, `spearman_rho`) et les tester.
- Prédire puis mesurer l'intégrale d'un champ le long d'une boucle, comparer une moyenne marginale et une moyenne conditionnelle, tracer deux chemins dans un paysage, corriger trois erreurs, écrire trois tests de propriété, et relever un défi.

**Rappel express.** La valeur de Shapley : $\varphi_i = \sum_{S \not\ni i}w(s)\big[v(S \cup \{i\}) - v(S)\big]$, avec $w(s) = \frac{s!\,(n - 1 - s)!}{n!}$, le poids d'**une** coalition de taille $s$ ; dans un vote pondéré $[q ;\ \omega_1, \ldots, \omega_n]$, $i$ est pivot quand $S$ perd et $S \cup \{i\}$ gagne. Les dividendes : $d_T = \sum_{S \subseteq T}(-1)^{\lvert T\rvert - \lvert S\rvert}v(S)$ et $\varphi_i = \sum_{T \ni i}\frac{d_T}{\lvert T\rvert}$. Le long d'un chemin $\boldsymbol\gamma$ de paramètre $t \in [0, 1]$ : $\int_{\boldsymbol\gamma}\mathbf{F}\cdot\mathrm{d}\boldsymbol\gamma = \int_0^1\mathbf{F}(\boldsymbol\gamma(t))\cdot\boldsymbol\gamma'(t)\,\mathrm{d}t$, et $\int_{\boldsymbol\gamma}\nabla f\cdot\mathrm{d}\boldsymbol\gamma = f(\text{fin}) - f(\text{début})$. Spearman : la corrélation de Pearson des rangs, rangs moyens aux ex aequo. Fixer une variable : $\operatorname{PD}_j(v) = \mathbb{E}[f(v, \mathbf{X}_{-j})]$ ou $M_j(v) = \mathbb{E}[f(\mathbf{X}) \mid X_j = v]$.

## Partie 0 · Vérifier tes exercices papier (✏️ MB6.1 à MB6.7 ; 🔗 MB6.11)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]`, dans l'ordre que dit la question.

Les réponses pas encore remplies affichent ⏳.

**Ex MB6.1 — Un conseil de quatre membres à vote pondéré : l'indice de Shapley-Shubik par les 24 ordres, un membre nul, deux membres de poids différents et de même pouvoir, et le nombre d'ordres tirés au hasard qui suffit (Hoeffding)**

In [ ]:
# Ex MB6.1: the values you found on paper
answer_MB6_1a = ...  # a) a list of 4 integers: the number of orders in which A, B, C, D is pivot
answer_MB6_1b = ...  # b) a list of 4 numbers (4 decimals): the indices of A, B, C, D
answer_MB6_1c = ...  # c) a list of 4 integers: the pivots when D has 2 votes
answer_MB6_1d = ...  # d) an integer: the smallest number of orders m

for letter, answer in zip("abcd", [answer_MB6_1a, answer_MB6_1b, answer_MB6_1c, answer_MB6_1d]):
    wbm.check(f"MB6.1{letter}", answer)

**Ex MB6.2 — Trois hameaux partagent une conduite d'eau : la valeur de Shapley par les ordres, les dividendes de Harsanyi, le partage tronçon par tronçon, et les quatre axiomes vérifiés**

In [ ]:
# Ex MB6.2: the values you found on paper
answer_MB6_2a = ...  # a) a list of 3 numbers (1 decimal): the shares of H1, H2, H3
answer_MB6_2b = ...  # b) a list of 3 numbers: the dividends of {H1, H2}, {H1, H3}, {H2, H3}
answer_MB6_2c = ...  # c) a number: the dividend of the three hamlets together
answer_MB6_2d = ...  # d) a list of 3 numbers (1 decimal): the new shares

for letter, answer in zip("abcd", [answer_MB6_2a, answer_MB6_2b, answer_MB6_2c, answer_MB6_2d]):
    wbm.check(f"MB6.2{letter}", answer)

**Ex MB6.3 — Le rendement d'une parcelle selon la pluie et l'engrais : courbes ICE et dérivée croisée de deux modèles, puis la moyenne marginale et la moyenne conditionnelle quand pluie et engrais sont corrélés**

In [ ]:
# Ex MB6.3: the values you found on paper
answer_MB6_3a = ...  # a) a list of 3 numbers (1 decimal): model 1, the gain for the three plots
answer_MB6_3b = ...  # b) a list of 3 numbers (1 decimal): model 2, the same gain
answer_MB6_3c = ...  # c) a list of 2 numbers: the cross derivative of model 1, then of model 2
answer_MB6_3d = ...  # d) a list of 2 numbers (3 decimals): PD(2), then M(2)
answer_MB6_3e = ...  # e) a number (3 decimals): the slope of M(v) − PD(v)

for letter, answer in zip("abcde", [answer_MB6_3a, answer_MB6_3b, answer_MB6_3c, answer_MB6_3d, answer_MB6_3e]):
    wbm.check(f"MB6.3{letter}", answer)

**Ex MB6.4 — Le travail de deux forces entre deux points : un champ de gradient le long de trois chemins, puis un champ qui n'en est pas un (Schwarz) le long de deux chemins**

In [ ]:
# Ex MB6.4: the values you found on paper
answer_MB6_4a = ...  # a) a number: the work of F along the segment
answer_MB6_4b = ...  # b) a list of 2 numbers: the work of F along the horizontal leg, then the vertical leg
answer_MB6_4c = ...  # c) a number: the work of F along the arc of parabola
answer_MB6_4d = ...  # d) a list of 2 numbers: dG1/dy and dG2/dx at A
answer_MB6_4e = ...  # e) a list of 2 numbers: the work of G along the segment, then the staircase

for letter, answer in zip("abcde", [answer_MB6_4a, answer_MB6_4b, answer_MB6_4c, answer_MB6_4d, answer_MB6_4e]):
    wbm.check(f"MB6.4{letter}", answer)

**Ex MB6.5 — La variation d'un coût à trois facteurs (prix, quantité, taux de change) partagée entre les facteurs : le long du segment, puis par la moyenne des six chemins en escalier, et pourquoi les deux partages coïncident**

In [ ]:
# Ex MB6.5: the values you found on paper
answer_MB6_5a = ...  # a) a number (2 decimals): C(x) − C(r), in euros
answer_MB6_5b = ...  # b) a list of 3 numbers (2 decimals): the shares of the price, the quantity, the exchange rate
answer_MB6_5c = ...  # c) a list of 2 numbers (2 decimals): the exchange rate in the order (p, q, tau), then (tau, p, q)
answer_MB6_5d = ...  # d) a number (2 decimals): the mean contribution of the exchange rate

for letter, answer in zip("abcd", [answer_MB6_5a, answer_MB6_5b, answer_MB6_5c, answer_MB6_5d]):
    wbm.check(f"MB6.5{letter}", answer)

**Ex MB6.6 — Des relevés de capteurs sur une grille 3 × 4 : la dérivée d'un indice régional par un réchauffement uniforme, puis la grille agrandie deux fois par interpolation bilinéaire, centres ou coins alignés**

In [ ]:
# Ex MB6.6: the values you found on paper
answer_MB6_6a = ...  # a) a number (1 decimal): the index y
answer_MB6_6b = ...  # b) an integer: the derivative by a uniform warming
answer_MB6_6c = ...  # c) a number (1 decimal): the index after a warming of 1 °C
answer_MB6_6d = ...  # d) a list of 6 numbers (2 decimals): centres aligned, the coordinates of the 6 rows
answer_MB6_6e = ...  # e) a list of 6 numbers (2 decimals): corners aligned, the coordinates of the 6 rows
answer_MB6_6f = ...  # f) a list of 2 numbers (2 decimals): the cell (row 4, column 5), centres aligned, then corners aligned

for letter, answer in zip("abcdef", [answer_MB6_6a, answer_MB6_6b, answer_MB6_6c, answer_MB6_6d, answer_MB6_6e, answer_MB6_6f]):
    wbm.check(f"MB6.6{letter}", answer)

**Ex MB6.7 — Un contrôle de pièces sur deux lignes de production aux taux de défauts différents : les taux de rejet sous la séparation, un tirage au sort entre deux seuils pour atteindre un taux de détection, et les poids qui rendent ligne et défaut indépendants**

In [ ]:
# Ex MB6.7: the values you found on paper
answer_MB6_7a = ...  # a) a list of 2 numbers (4 decimals): the rejection rate of each line
answer_MB6_7b = ...  # b) a number (4 decimals): the probability q
answer_MB6_7c = ...  # c) a number (4 decimals): the rate of false alarms obtained
answer_MB6_7d = ...  # d) a list of 4 numbers (4 decimals): w(1, defect), w(1, no defect), w(2, defect), w(2, no defect)
answer_MB6_7e = ...  # e) a list of 2 numbers (4 decimals): the weighted defect rate of each line

for letter, answer in zip("abcde", [answer_MB6_7a, answer_MB6_7b, answer_MB6_7c, answer_MB6_7d, answer_MB6_7e]):
    wbm.check(f"MB6.7{letter}", answer)

**Ex MB6.11 — Pont : l'écart des taux de détection de deux laboratoires et son intervalle, par l'erreur type d'une différence de deux proportions, qui se lit avec le nombre d'échantillons positifs de chacun**

In [ ]:
# Ex MB6.11: the values you found on paper
answer_MB6_11a = ...  # a) a list of 2 numbers (4 decimals): the detection rates of A and B
answer_MB6_11b = ...  # b) a number (4 decimals): the standard error of TPR_B − TPR_A
answer_MB6_11c = ...  # c) a list of 2 numbers (3 decimals): the Wald interval at 95 %
answer_MB6_11d = ...  # d) a number (4 decimals): the standard error with 4 times more contaminated samples in A

for letter, answer in zip("abcd", [answer_MB6_11a, answer_MB6_11b, answer_MB6_11c, answer_MB6_11d]):
    wbm.check(f"MB6.11{letter}", answer)

## Partie A · Ta librairie : les poids des coalitions, les dividendes de Harsanyi, l'intégrale le long d'une ligne brisée, la corrélation de Spearman

Fiche §1, §2, §4 et §6. Les quatre fonctions de `mymath/attribution.py`, leurs tests, et un usage de chacune. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import contextlib
import dis
import math
import os
import re
import sys
import time
from fractions import Fraction
from types import MappingProxyType

import matplotlib.pyplot as plt
import numpy as np

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, refuse_peeking, run_pytest_group, verdict

TEST_MB6 = "tests/test_mb6_attribution.py"           # the tests of mymath/attribution.py
REF = wbm.load_mymath("ref", alias="mymath_ref")       # the reference: some checks of the hidden cases rely on it
ref_att = REF.attribution


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())[:160]
    except (TypeError, ValueError):
        return repr(value)[:160]


GENERIC_COMPUTED = "Ta fonction, ou le calcul qui l'utilise, ne donne pas la bonne valeur ici."


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) or (
        isinstance(v, (list, tuple)) and any(isinstance(w, (bool, np.bool_)) for w in v)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown") and "Piste" not in message:
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            if message.startswith(GENERIC_COMPUTED):            # one text, not two: the hint says what to look at
                message = GENERIC_COMPUTED
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def library_hint(names):
    """A hint for a wrong answer computed with the learner's library (no classic error recognised): its tests first,
    then the call."""
    return lambda value: (f"si tu t'es servi de {names} de ta librairie et que ses tests passent tous (plus haut), "
                          "l'erreur est dans l'appel (les arguments, leur ordre, ce que tu lis dans le résultat) ; et "
                          "donne la valeur que rend le calcul, sans l'arrondir")


def passes(ex_id, value):
    """True when value is the hidden answer ex_id (wbm.check, computed=True, nothing printed)."""
    try:
        return bool(wbm.check(ex_id, value, computed=True, quiet=True).passed)
    except Exception:  # noqa: BLE001 - an odd value is not the answer
        return False


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def call_safely(label, function, *args, **kwargs):
    """function(*args, **kwargs); an error of the learner's code becomes one ❌ line naming the call (StopCheck)."""
    try:
        return function(*args, **kwargs)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the call
        print(f"❌ {label} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        raise StopCheck from None


def frozen(array):
    """A float copy of array that refuses any change in place (the data given to the learner's functions)."""
    out = np.array(array, dtype=float)
    out.setflags(write=False)
    return out


class CodeFrom(BaseException):
    """Raised inside code_from when watched code starts (a BaseException: a broad except of the learner's code does
    not hide it)."""


_NOT_KEYED = {"CACHE", "PUSH_NULL", "RESUME", "NOP", "EXTENDED_ARG", "NOT_TAKEN"}


def code_key(code):
    """What identifies a compiled function whatever its file and the module around it: its instructions (without jump
    targets, nor what depends on the rest of the module) and the names of its variables."""
    ops = []
    for i in dis.get_instructions(code):
        if i.opname in _NOT_KEYED:
            continue
        jump = i.opcode in dis.hasjrel or i.opcode in dis.hasjabs or "JUMP" in i.opname or i.opname == "FOR_ITER"
        ops.append((i.opname, None if jump else i.argval.co_name if hasattr(i.argval, "co_code") else repr(i.argval)))
    return tuple(ops), code.co_varnames


def code_keys_of_file(path):
    """The code_key of every function of a Python file, compiled but not run (to recognise a copy of its code); an
    empty set for a file that cannot be read or compiled."""
    try:
        keys, todo = set(), [compile(open(path, encoding="utf-8").read(), str(path), "exec")]
    except (OSError, SyntaxError, ValueError, UnicodeDecodeError):
        return set()
    while todo:
        code = todo.pop()
        todo += [c for c in code.co_consts if hasattr(c, "co_code")]
        key = code_key(code)
        if code.co_name != "<module>" and len(key[0]) >= 15:
            keys.add(key)
    return keys


_ESCAPE = {"watch": None}      # what code_from(..., escapes=True) watches now (None: nothing)
_PROCESS_EVENTS = ("subprocess.Popen", "os.system", "os.posix_spawn", "os.fork", "os.forkpty", "os.exec", "os.spawn")


def _escape_hook(event, args, escape=_ESCAPE, processes=_PROCESS_EVENTS, os=os, stop=CodeFrom):
    """Audit hook, for the life of the kernel: inside code_from(..., escapes=True), opening a file of the watched places
    or starting another process stops the block (code run by another process escapes sys.monitoring). Everything it
    uses is bound when it is defined: it also runs while the kernel shuts down."""
    watch = escape["watch"]
    if watch is None:
        return
    if event == "open" and args and isinstance(args[0], (str, bytes, os.PathLike)):
        real = os.path.realpath(os.fsdecode(args[0]))
        if any(real == place or real.startswith(place + os.sep) for place in watch["places"]):
            watch["seen"].append("ouvre " + os.path.basename(real))
            raise stop
    elif event.startswith(processes):
        watch["seen"].append("lance un autre processus : " + event)
        raise stop


sys.addaudithook(_escape_hook)


@contextlib.contextmanager
def code_from(places, twins=(), escapes=False):
    """Inside the block, the start of any Python code (a function or a module body) of a file among places (files, or
    folders with everything under them), or of a function whose code_key is among twins (a copy of such a function),
    stops the block; yields the list of the names seen ("attribution.coalition_weights"). The learner's function must
    not run that code (mymath in a corrected function, the hidden module of a challenge). With escapes, opening a file
    of places or starting another process also stops it ("ouvre mb6_22.py", "lance un autre processus :
    subprocess.Popen"). Without sys.monitoring (Python < 3.12) or a free tool id, code is not watched."""
    places = [os.path.realpath(str(place)) for place in places]
    libraries = tuple({os.path.realpath(sys.prefix) + os.sep, os.path.realpath(sys.base_prefix) + os.sep})
    monitoring, tool, seen, inside = getattr(sys, "monitoring", None), None, [], {}
    for candidate in ((4, 3, 2) if monitoring is not None else ()):
        try:
            monitoring.use_tool_id(candidate, "code_from")
        except ValueError:                               # taken by another tool
            continue
        tool = candidate
        break
    if tool is not None:
        def on_start(code, offset):
            name = code.co_filename
            if name not in inside:
                real = os.path.realpath(name) if os.path.exists(name) else name
                if any(real == place or real.startswith(place + os.sep) for place in places):
                    inside[name] = True                  # watched
                elif twins and not real.startswith(libraries):
                    inside[name] = False                 # the learner's code: compared with twins
                else:
                    inside[name] = None                  # an installed library: not compared
            if inside[name]:
                seen.append(f"{os.path.splitext(os.path.basename(name))[0]}.{code.co_name}")
                raise CodeFrom
            if inside[name] is False and code_key(code) in twins:
                seen.append(f"une copie de {code.co_name}")
                raise CodeFrom
            return monitoring.DISABLE                    # other code: not watched again until the end of the block

        monitoring.register_callback(tool, monitoring.events.PY_START, on_start)
        monitoring.set_events(tool, monitoring.events.PY_START)
    if escapes:
        _ESCAPE["watch"] = {"places": places, "seen": seen}
    try:
        yield seen
    except CodeFrom:
        pass
    finally:
        _ESCAPE["watch"] = None
        if tool is not None:
            monitoring.set_events(tool, 0)
            monitoring.register_callback(tool, monitoring.events.PY_START, None)
            monitoring.free_tool_id(tool)
            monitoring.restart_events()


# The data of the paper exercises (02_exercices.md), read only
COUNCIL_1 = MappingProxyType({"quota": 11, "votes": (8, 6, 3, 1)})        # MB6.1: the votes of A, B, C, D (0 to 3)
PIPE_2 = MappingProxyType({"sections": (30, 21, 17), "needs": (30, 51, 68)})   # MB6.2: k€; needs = the pipe up to
#   each hamlet H1, H2, H3 (0 to 2)
YIELD_3 = MappingProxyType({"plots": (4.0, 5.5, 7.0), "mean_p": 5.5, "sd_p": 1.2, "mean_e": 1.5, "sd_e": 0.5,
                            "rho": 0.35})                                 # MB6.3: rain p and fertiliser e
FORCES_4 = MappingProxyType({"A": (1.0, 0.0), "B": (2.0, 1.0), "corner": (2.0, 0.0)})   # MB6.4: m; the staircase
#   goes from A to the corner, then to B
COST_5 = MappingProxyType({"r": (11.5, 680.0, 0.93), "x": (13.5, 590.0, 0.87)})   # MB6.5: (price $, quantity, €/$)
GRID_6 = MappingProxyType({"T": frozen([[11.8, 9.4, 13.2, 10.8], [8.7, 12.5, 14.1, 9.9], [10.3, 7.6, 12.0, 11.4]]),
                           "a": frozen([[2, 1, 3, 1], [1, 2, 2, 1], [3, 1, 1, 2]])})   # MB6.6: °C, hundreds of ha
LINES_7 = MappingProxyType({"defect_rates": (0.06, 0.14), "tpr": 0.88, "fpr": 0.04, "threshold_A": (0.03, 0.74),
                            "threshold_B": (0.09, 0.96), "target": 0.90, "pieces": (1300, 1100),
                            "defective": (78, 154)})                      # MB6.7: thresholds as (fpr, tpr), line 2
LABS_11 = MappingProxyType({"contaminated": (28, 520), "signalled": (20, 448)})   # MB6.11: A, B

### Ex MB6.13 — Les poids des coalitions dans la valeur de Shapley, exacts ou en flottants · `coalition_weights` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer les poids des coalitions dans la valeur de Shapley, exacts ou en flottants, puis voir combien une coalition de taille moyenne pèse peu.  
**Prérequis :** fiche §1 ; R8 (factorielles, combinaisons) · **mymath :** `attribution.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en MB5). Ouvre ta copie `mon_travail/mymath/attribution.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.attribution.<nom>` (par exemple `mymath.attribution.spearman_rho(x, y)`), ou donne-leur un nom court (`spearman_rho = mymath.attribution.spearman_rho`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import itertools`, `import math`, `from fractions import Fraction` et `import numpy as np`, et `from .integrate import simpson` si tu reprends ta règle de Simpson de M02a. NumPy et les modules `math`, `fractions` et `itertools` sont permis, mais ni SciPy (`rankdata`, `spearmanr`, `pearsonr`, `scipy.integrate`), ni `np.corrcoef`, ni `np.cov`, ni le module `statistics`, ni pandas : les tests comparent tes résultats à `math.comb`, à `fractions`, à `scipy.integrate.quad` et à `scipy.stats.spearmanr`, et un test lit ton module. Ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `coalition_weights(n_players, exact=False)` (fiche §1), en suivant sa docstring : contrôle les arguments (`n_players` un entier $\ge 1$, pas un booléen ni un flottant ; `exact` un booléen), puis rends les $n$ poids $w(0), \ldots, w(n - 1)$, en `Fraction` (une liste) ou en flottants (un tableau NumPy). Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) les 9 poids pour $n = 9$, exacts (une liste de 9 fractions : passe `exact=True`) ;
- b) pour $n = 25$, le conseil du 🏆, le rapport $\frac{w(0)}{w(12)}$ (un entier).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB6")  # reload your saved file

**Les tests** de `coalition_weights` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB6, "test_coalition_weights_ or test_module_uses_")

**L'usage** : les poids d'un jeu de 9 joueurs, puis un rapport de poids pour le conseil de 25 membres.

In [ ]:
answer_MB6_13a = answer_MB6_13b = ...
with wbm.attempt("MB6.13"):          # if you call coalition_weights here: ⏳ while it is not written
    answer_MB6_13a = ...  # a) a list of 9 fractions: the exact weights w(0), ..., w(8) for n = 9
    answer_MB6_13b = ...  # b) an integer: w(0) / w(12) for n = 25

In [ ]:
def exact_13(value):
    """None when a) holds exact numbers (Fractions, or fractions written as text); a message when it holds floats."""
    try:
        items = list(value)
    except TypeError:
        return None
    if any(isinstance(item, (float, np.floating)) for item in items):
        return ("a) demande les poids exacts : des fractions (des Fraction, avec exact=True), pas des flottants, qui en "
                "sont des valeurs approchées")
    return None


if filled(answer_MB6_13a) and exact_13(answer_MB6_13a):
    print(f"❌ Ex MB6.13a : {exact_13(answer_MB6_13a)}")
else:
    check_answer("MB6.13a", answer_MB6_13a, hint=library_hint("coalition_weights"))
check_answer("MB6.13b", answer_MB6_13b, hint=library_hint("coalition_weights"))

**Ce qu'il faut regarder** : pourquoi une coalition de taille moyenne pèse-t-elle si peu, alors que chaque taille pèse $\frac1n$ en tout ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB6.14 — Les dividendes de Harsanyi d'un jeu donné par sa fonction caractéristique · `harsanyi_dividends` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer les dividendes de Harsanyi d'un jeu donné par sa fonction caractéristique, puis en tirer la valeur de Shapley et lire ce que dit un dividende négatif.  
**Prérequis :** fiche §2 ; R4 (parties d'un ensemble) ; Ex MB6.1, MB6.2 · **mymath :** `attribution.py` · **Parcours :** C

> Même mode d'emploi qu'en MB6.13 : écris la fonction dans ta copie `mon_travail/mymath/attribution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `harsanyi_dividends(v, n_players)` (fiche §2), en suivant sa docstring : contrôle les arguments ; appelle `v` une fois par coalition (taille par taille, avec `itertools.combinations`) et garde les valeurs ; puis, pour chaque $T$, la somme alternée sur ses parties ; rends le dictionnaire dans l'ordre de la docstring. Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) les hameaux de MB6.2 (`PIPE_2`, cellule de données ; numérotés 0, 1, 2, besoins 30, 51 et 68) : $d_{\{0, 1\}}$, $d_{\{0, 2\}}$, $d_{\{1, 2\}}$, $d_{\{0, 1, 2\}}$ (une liste de 4 nombres, à comparer à ta copie) ;
- b) quatre maraîchers partagent un stand au marché ; le notebook fournit `TABLE_14`, le bénéfice hebdomadaire $v(S)$ de chaque coalition (en centaines d'euros), un dictionnaire indexé par des `frozenset` : leurs valeurs de Shapley par $\varphi_i = \sum_{T \ni i}\frac{d_T}{\lvert T\rvert}$ (une liste de 4 nombres) ;
- c) $d_{\{1, 2\}}$ et $d_{\{0, 1, 3\}}$ du stand (une liste de 2 nombres) ;
- d) le conseil de MB6.1 (`COUNCIL_1` ; A, B, C, D numérotés 0 à 3) : $d_{\{0, 1\}}$, $d_{\{0, 2\}}$, $d_{\{0, 1, 2\}}$ (une liste de 3 nombres).

In [ ]:
# Four market gardeners (0 to 3) share a stand: the weekly profit v(S) of each coalition (hundreds of euros), read only
TABLE_14 = MappingProxyType({frozenset(members): worth for members, worth in [((), 0.0), ((0,), 3.0), ((1,), 5.0), ((2,), 2.0), ((3,), 4.0), ((0, 1), 10.0), ((0, 2), 5.0), ((0, 3), 7.0),
 ((1, 2), 6.0), ((1, 3), 9.0), ((2, 3), 9.0), ((0, 1, 2), 11.0), ((0, 1, 3), 15.5), ((0, 2, 3), 12.0),
 ((1, 2, 3), 13.0), ((0, 1, 2, 3), 19.5)]})

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB6")  # reload your saved file

**Les tests** de `harsanyi_dividends` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB6, "test_harsanyi_dividends_")

**L'usage** : les hameaux de MB6.2, le stand des maraîchers, puis le conseil de MB6.1.

In [ ]:
answer_MB6_14a = answer_MB6_14b = answer_MB6_14c = answer_MB6_14d = ...
with wbm.attempt("MB6.14"):          # if you call harsanyi_dividends here: ⏳ while it is not written
    answer_MB6_14a = ...  # a) a list of 4 numbers, as computed: the hamlets, d_{0,1}, d_{0,2}, d_{1,2}, d_{0,1,2}
    answer_MB6_14b = ...  # b) a list of 4 numbers, as computed: the Shapley values of the four gardeners
    answer_MB6_14c = ...  # c) a list of 2 numbers, as computed: d_{1,2} and d_{0,1,3} of the stand
    answer_MB6_14d = ...  # d) a list of 3 numbers, as computed: the council, d_{0,1}, d_{0,2}, d_{0,1,2}

In [ ]:
check_answer("MB6.14a", answer_MB6_14a, hint=library_hint("harsanyi_dividends"))
check_answer("MB6.14b", answer_MB6_14b, hint=library_hint("harsanyi_dividends"))
check_answer("MB6.14c", answer_MB6_14c, hint=library_hint("harsanyi_dividends"))
check_answer("MB6.14d", answer_MB6_14d, hint=library_hint("harsanyi_dividends"))

**Ce qu'il faut regarder** : quel dividende du stand est négatif, et que dit-il des maraîchers 1 et 2 ? Dans le conseil, pourquoi $d_{\{0, 1, 2\}}$ est-il négatif ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB6.15 — L'intégrale d'un champ de vecteurs le long d'une ligne brisée, par la méthode de Simpson sur chaque segment · `line_integral` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer l'intégrale d'un champ de vecteurs le long d'une ligne brisée par la méthode de Simpson, puis retrouver MB6.4 et approcher un arc par des cordes.  
**Prérequis :** fiche §4 ; M02a (méthode de Simpson) ; Ex MB6.4 · **mymath :** `attribution.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB6.13 : écris la fonction dans ta copie `mon_travail/mymath/attribution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `line_integral(field, points, n_sub=20)` (fiche §4), en suivant sa docstring : contrôle les arguments (des points finis, deux dimensions, au moins deux lignes ; `n_sub` un entier pair $\ge 2$ ; ce que rend `field`) ; pour chaque segment de $\mathbf{a}$ à $\mathbf{b}$, la règle de Simpson (M02a) sur les $n_{\text{sub}} + 1$ valeurs de $t \mapsto \mathbf{F}(\mathbf{a} + t(\mathbf{b} - \mathbf{a}))\cdot(\mathbf{b} - \mathbf{a})$, en appelant `field` point par point ; la somme sur les segments. À défaut de `integrate.py` (M02a), une règle de Simpson privée dans `attribution.py` convient. Lance le groupe de tests.

**L'usage.** Écris d'abord `F_15(p)` et `G_15(p)`, les champs $\mathbf{F} = \nabla f = (2xy^2 + 5,\ 2x^2y)$, $f(x, y) = x^2y^2 + 5x$, et $\mathbf{G}(x, y) = (2y, 3x^2)$ de MB6.4, en un point `p` $= (x, y)$ (la vérification les essaie en des points cachés). Puis, avec ta fonction et les points de `FORCES_4` (cellule de données) :
- a) $\mathbf{F}$ le long du segment $[AB]$, puis de l'escalier $A \to (2, 0) \to B$ (une liste de 2 nombres, à comparer à ta copie) ;
- b) $\mathbf{G}$ le long des deux mêmes chemins (une liste de 2 nombres) ;
- c) $\mathbf{G}$ le long de lignes brisées qui suivent l'arc $\boldsymbol\gamma(t) = (1 + t, t^2)$, avec 4, 16 et 64 segments de même pas en $t$ (une liste de 3 nombres ; l'intégrale de $\mathbf{G}$ le long de l'arc vaut $\frac{55}{6}$).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB6")  # reload your saved file

**Les tests** de `line_integral` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB6, "test_line_integral_")

**L'usage** : les deux forces de MB6.4, puis des cordes de plus en plus nombreuses le long d'un arc.

In [ ]:
def F_15(p):
    """The field F of MB6.4, the gradient of f(x, y) = x² y² + 5x, at the point p = (x, y)."""
    raise NotImplementedError("F_15 n'est pas encore écrite (04_indices.md si tu bloques)")


def G_15(p):
    """The field G(x, y) = (2y, 3x²) of MB6.4, at the point p = (x, y)."""
    raise NotImplementedError("G_15 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB6_15a = answer_MB6_15b = answer_MB6_15c = ...
with wbm.attempt("MB6.15"):          # your fields and line_integral: ⏳ while they are not written
    answer_MB6_15a = ...  # a) a list of 2 numbers, as computed: F along the segment [AB], then along the staircase
    answer_MB6_15b = ...  # b) a list of 2 numbers, as computed: G along the same two paths
    answer_MB6_15c = ...  # c) a list of 3 numbers, as computed: G along the polylines of 4, 16 and 64 segments on the arc

In [ ]:
HIDDEN_C15 = {'points': [(1.37, -0.62), (-0.83, 1.94), (2.41, 0.29), (0.56, -1.73)],
 'id_F_15': 'MB6.15d',
 'id_G_15': 'MB6.15e'}   # hidden points, and the sub-IDs of the values of F_15 and G_15 there

fields_ok_15 = {}
with checked("MB6.15"):
    for name_c15, field_c15 in (("F_15", F_15), ("G_15", G_15)):
        got_c15 = []
        for point_c15 in HIDDEN_C15["points"]:
            value_c15 = as_floats(call_safely(f"Ex MB6.15 : {name_c15} (appel caché)", field_c15, np.array(point_c15)))
            if value_c15 is None or value_c15.shape != (2,):
                print(f"❌ Ex MB6.15 : {name_c15} rend un vecteur de 2 nombres en un point (x, y), reçu {shown(value_c15)}")
                raise StopCheck
            got_c15 += value_c15.tolist()
        fields_ok_15[name_c15] = passes(HIDDEN_C15["id_" + name_c15], got_c15)
    for name_c15, what_c15 in (("F_15", "le gradient de f(x, y) = x²y² + 5x"), ("G_15", "le champ G(x, y) = (2y, 3x²)")):
        print(f"✅ Ex MB6.15 : {name_c15}, en des points cachés : {what_c15}" if fields_ok_15[name_c15] else
              f"❌ Ex MB6.15 : {name_c15}, en des points cachés : ce n'est pas {what_c15} (MB6.4)")


def hint_15(value):
    """A targeted hint for a) to c): the fields first, then the library."""
    wrong = [name for name in ("F_15", "G_15") if not fields_ok_15.get(name, True)]
    if wrong:
        return f"les intégrales se calculent avec {' et '.join(wrong)} : corrige d'abord la ligne ❌ ci-dessus"
    return library_hint("line_integral")(value)


check_answer("MB6.15a", answer_MB6_15a, hint=hint_15)
check_answer("MB6.15b", answer_MB6_15b, hint=hint_15)
check_answer("MB6.15c", answer_MB6_15c, hint=hint_15)

**Ce qu'il faut regarder** : la ligne brisée s'approche-t-elle de $\frac{55}{6}$ quand les segments se multiplient ? Pourquoi $\mathbf{F}$ donne-t-il la même valeur pour toutes ces lignes brisées ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB6.16 — La corrélation de rangs de Spearman, ex aequo compris · `spearman_rho` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la corrélation de rangs de Spearman, ex aequo compris, puis comparer deux échelles de dureté.  
**Prérequis :** fiche §6 ; M02c (corrélation de Pearson) · **mymath :** `attribution.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB6.13 : écris la fonction dans ta copie `mon_travail/mymath/attribution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `spearman_rho(x, y)` (fiche §6), en suivant sa docstring : contrôle les arguments ; les rangs de chaque échantillon (`np.argsort`, puis le rang moyen de chaque groupe de valeurs égales, comptés à partir de 1) ; la formule de Pearson sur les rangs centrés ; `nan` quand une somme de carrés est nulle. Lance le groupe de tests.

**L'usage.** Le notebook fournit `MOHS_16` et `VICKERS_16`, la dureté de neuf échantillons de minéraux sur l'échelle de Mohs (des degrés et des demi-degrés, avec un ex aequo) et par un essai de Vickers (en HV). Avec ta fonction :
- a) $\rho_S$(Mohs, Vickers) (un nombre) ;
- b) $\rho_S$(Mohs, $\frac{1}{\text{Vickers}}$) (un nombre).

In [ ]:
# The hardness of nine samples of minerals: on Mohs's scale (degrees and half degrees) and by a Vickers test (HV)
MOHS_16 = frozen([2, 3, 3, 4, 5, 5.5, 6, 7, 8])
VICKERS_16 = frozen([32, 110, 150, 600, 540, 190, 1400, 800, 1100])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB6")  # reload your saved file

**Les tests** de `spearman_rho` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB6, "test_spearman_rho_")

**L'usage** : deux échelles de dureté des mêmes minéraux.

In [ ]:
answer_MB6_16a = answer_MB6_16b = ...
with wbm.attempt("MB6.16"):          # if you call spearman_rho here: ⏳ while it is not written
    answer_MB6_16a = ...  # a) a number, as computed: rho_S(Mohs, Vickers)
    answer_MB6_16b = ...  # b) a number, as computed: rho_S(Mohs, 1 / Vickers)

In [ ]:
check_answer("MB6.16a", answer_MB6_16a, hint=library_hint("spearman_rho"))
check_answer("MB6.16b", answer_MB6_16b, hint=library_hint("spearman_rho"))

**Ce qu'il faut regarder** : pourquoi la formule $1 - \frac{6\sum d_i^2}{n(n^2 - 1)}$ n'est-elle ici qu'une approximation ? Pourquoi b) se déduit-il de a) sans calcul ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie B · Prédire, mesurer, dessiner

Fiche §3 et §4. Un champ de gradient et un champ tournant le long d'une boucle ; la moyenne marginale et la moyenne conditionnelle de deux mesures corrélées ; deux chemins dans un paysage.

### Ex MB6.17 — Prédire l'intégrale d'un champ de gradient et d'un champ tournant le long d'une boucle triangulaire, dans un sens, dans l'autre et deux fois, puis vérifier 🔮 ★ ⏱️ 15 min
**Objectif :** prédire l'intégrale d'un champ de gradient et d'un champ tournant le long d'une boucle, puis la mesurer.  
**Prérequis :** fiche §4 ; Ex MB6.4, MB6.15 · **Parcours :** C

Le triangle $A = (1, 0)$, $B = (4, 1)$, $C = (2, 3)$ (`TRIANGLE_17`), et deux champs : $\mathbf{F}$ de MB6.4 (un gradient, ton `F_15`) et le champ tournant $\mathbf{R}(x, y) = (-y, x)$. **Avant d'exécuter quoi que ce soit**, prédis (une lettre chacune) :
- a) l'intégrale de $\mathbf{F}$ le long de la boucle $A \to B \to C \to A$ : (A) 0 ; (B) l'aire du triangle ; (C) elle dépend du triangle choisi ;
- b) celle de $\mathbf{R}$ le long de $A \to B \to C \to A$ : (A) 0 ; (B) l'aire du triangle ; (C) deux fois l'aire du triangle ;
- c) celle de $\mathbf{R}$ le long de $A \to C \to B \to A$ : (A) la même qu'en b) ; (B) l'opposée de b) ; (C) 0 ;
- d) celle de $\mathbf{R}$ quand on fait deux fois le tour $A \to B \to C \to A$ : (A) la même qu'en b) ; (B) le double de b) ; (C) le carré de b).

Puis, une fois tes prédictions écrites et vérifiées, écris `R_17(p)`, le champ tournant en un point `p` $= (x, y)$, et mesure avec `line_integral` :
- e) les quatre intégrales de a) à d), dans cet ordre (une liste de 4 nombres).

La vérification de l'expérience ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
TRIANGLE_17 = ((1.0, 0.0), (4.0, 1.0), (2.0, 3.0))     # the vertices A, B, C of the triangle

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_MB6_17a = ...  # a) "A", "B" or "C": F along A -> B -> C -> A: (A) 0, (B) the area, (C) it depends on the triangle
prediction_MB6_17b = ...  # b) "A", "B" or "C": R along A -> B -> C -> A: (A) 0, (B) the area, (C) twice the area
prediction_MB6_17c = ...  # c) "A", "B" or "C": R along A -> C -> B -> A: (A) the same as b), (B) the opposite of b), (C) 0
prediction_MB6_17d = ...  # d) "A", "B" or "C": R twice around A -> B -> C -> A: (A) the same as b), (B) twice b), (C) b) squared

In [ ]:
for letter, prediction in zip("abcd", [prediction_MB6_17a, prediction_MB6_17b, prediction_MB6_17c,
                                      prediction_MB6_17d]):
    if filled(prediction) and not isinstance(prediction, str):      # a number, a list...: not yet a prediction
        print(f'❌ Ex MB6.17{letter} : une prédiction est une lettre, "A", "B" ou "C", reçu {shown(prediction)}')
    else:
        wbm.check(f"MB6.17{letter}", prediction)

**L'expérience.** Écris-la une fois tes quatre prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
def R_17(p):
    """The rotating field R(x, y) = (-y, x), at the point p = (x, y)."""
    raise NotImplementedError("R_17 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB6_17e = ...
with wbm.attempt("MB6.17"):          # your R_17, F_15 and line_integral: ⏳ while they are not written
    answer_MB6_17e = ...  # e) a list of 4 numbers, as computed: the integrals of a), b), c) and d), with line_integral

In [ ]:
def hint_17(value):
    """A targeted hint for e): the fields first, then the library."""
    if not globals().get("fields_ok_15", {}).get("F_15", True):
        return "e) se calcule avec F_15 de MB6.15 : corrige-la d'abord"
    return library_hint("line_integral")(value)


if any(answer is ... or answer is None for answer in [prediction_MB6_17a, prediction_MB6_17b, prediction_MB6_17c, prediction_MB6_17d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    check_answer("MB6.17e", answer_MB6_17e, hint=hint_17)

**Ce qu'il faut regarder** (après l'expérience) : l'aire du triangle vaut-elle ce que tu pensais ? Qu'est-ce qui, dans $\mathbf{R}$, tourne avec la boucle ? Une boucle d'intégrale nulle suffirait-elle à prouver qu'un champ est un gradient ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB6.18 — Deux mesures corrélées : la moyenne marginale (dépendance partielle) et la moyenne conditionnelle d'une même fonction, simulées pour plusieurs corrélations et comparées aux formules gaussiennes 🔬 ★★ ⏱️ 25 min
**Objectif :** comparer la moyenne marginale (la dépendance partielle) et la moyenne conditionnelle d'une même fonction de deux mesures corrélées, par les formules gaussiennes et par simulation.  
**Prérequis :** fiche §3 ; M25a (loi conditionnelle d'un couple gaussien), M11 (dispersion d'une moyenne) ; Ex MB6.3 · **Parcours :** C, E

En été, la température de l'eau $T$ (°C) et la concentration en phosphore $U$ (µg/L) d'un réservoir d'eau forment un couple gaussien : $T \sim \mathcal{N}(22, 3^2)$, $U \sim \mathcal{N}(70, 8^2)$, de corrélation $\rho$. Une fonction écrite donne la croissance relative des algues : $f(T, U) = \exp\big(0{,}1\,(T - 22) + 0{,}05\,(U - 70)\big)$ (`growth_18`). On fixe $T = 26$. Pour $\rho = 0$ ; $0{,}5$ et $0{,}9$ :
- a) calcule par les formules $\operatorname{PD}(26) = \mathbb{E}[f(26, U)]$ et $M(26) = \mathbb{E}[f(T, U) \mid T = 26]$ (une liste de 4 nombres : $\operatorname{PD}(26)$, puis $M(26)$ pour les trois $\rho$) ; tu peux utiliser $\mathbb{E}[e^{cZ}] = e^{c\mu + c^2\sigma^2/2}$ pour $Z \sim \mathcal{N}(\mu, \sigma^2)$ ;
- b) le notebook fournit `SAMPLES_18`, 300 000 couples tirés pour chaque $\rho$ (`SAMPLES_18[rho]` rend les tableaux `T` et `U`) ; estime $\operatorname{PD}(26)$ par la moyenne des $f(26, U_i)$, et $M(26)$ par la moyenne des $f(T_i, U_i)$ sur les points où $\lvert T_i - 26\rvert \le 0{,}25$ (une liste de 6 nombres : pour chaque $\rho$, l'estimation de $\operatorname{PD}$ puis celle de $M$).

In [ ]:
# The reservoir in summer: the temperature T of the water (°C) and the phosphorus U (µg/L), a Gaussian couple of
# correlation rho; 300 000 draws for each rho (seed 7918), read only
def draws_18(rho, size, rng):
    """size draws of the couple (T, U): T ~ N(22, 3²), U ~ N(70, 8²), of correlation rho (two float arrays)."""
    pairs = rng.multivariate_normal([22.0, 70.0], [[9.0, 24.0 * rho], [24.0 * rho, 64.0]], size=size)
    return pairs[:, 0], pairs[:, 1]


_rng_18 = np.random.default_rng(7918)
SAMPLES_18 = MappingProxyType({rho: tuple(frozen(column) for column in draws_18(rho, 300_000, _rng_18))
                               for rho in (0.0, 0.5, 0.9)})          # rho: (T, U)
del _rng_18


def growth_18(T, U):
    """The relative growth of the algae, exp(0.1 (T - 22) + 0.05 (U - 70)) (numbers or arrays)."""
    return np.exp(0.1 * (np.asarray(T) - 22) + 0.05 * (np.asarray(U) - 70))

In [ ]:
answer_MB6_18a = ...  # a) a list of 4 numbers, by the formulas: PD(26), then M(26) for rho = 0, 0.5, 0.9
answer_MB6_18b = ...  # b) a list of 6 numbers, as computed on SAMPLES_18: for each rho, the estimate of PD(26), then of M(26)

In [ ]:
for letter, answer in zip("ab", [answer_MB6_18a, answer_MB6_18b]):
    check_answer(f"MB6.18{letter}", answer)

**Ce qu'il faut regarder** : pour quel $\rho$ les deux moyennes sont-elles égales ? Pourquoi l'estimation de $M$ est-elle plus dispersée que celle de $\operatorname{PD}$ ? Que deviendrait-elle avec une fenêtre dix fois plus étroite ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB6.19 — Les lignes de niveau d'une fonction de deux variables, deux chemins entre les mêmes points, et l'intégrale du gradient cumulée le long de chacun 🎨 ★ ⏱️ 25 min
**Objectif :** tracer les lignes de niveau d'une fonction, deux chemins entre les mêmes points, et l'intégrale du gradient cumulée le long de chacun.  
**Prérequis :** fiche §4 ; M05b (lignes de niveau) ; Ex MB6.15 · **Parcours :** C

Le notebook fournit `f_19(x, y)` $= 3e^{-((x - 1)^2 + (y - 1)^2)} - 2e^{-((x + 1)^2 + y^2)/2} + 0{,}2\,x$, son gradient `grad_19`, les extrémités $A$ = `A_19` $= (-2{,}5 ;\ -1)$ et $B$ = `B_19` $= (2, 2)$, et `CORNERS_19`. Écris `draw_19(ax_map, ax_cum)`, qui trace :
- dans `ax_map`, les lignes de niveau de $f$ sur $[-3, 3]^2$ (`ax_map.contour`) et deux chemins de $A$ à $B$, le segment et la ligne brisée $A \to (0, -2) \to (2{,}5 ;\ 0) \to B$, chacun par ses sommets, avec une légende qui nomme chaque chemin (« segment » et « ligne brisée ») ;
- dans `ax_cum`, pour chaque chemin, l'intégrale du gradient de $f$ depuis $A$ jusqu'au point courant (avec `line_integral` sur le début du chemin), en fonction de la longueur parcourue depuis $A$ (au moins 50 points par segment), avec des noms d'axes ;

et qui rend les deux tableaux de cumuls tracés (le segment, puis la ligne brisée). La fonction dessine dans les axes reçus, sans ouvrir de figure. La vérification l'appelle, affiche la figure, et compare tes cumuls à ceux de `line_integral` de la référence, à $0{,}005$ près. Relève :
- a) $f(B) - f(A)$ (un nombre) ;
- b) le plus grand cumul le long du segment, puis le long de la ligne brisée (une liste de 2 nombres, les maximums de tes deux tableaux).

In [ ]:
# The landscape of the exercise, its gradient, and the two ends of the paths
A_19, B_19 = (-2.5, -1.0), (2.0, 2.0)
CORNERS_19 = ((0.0, -2.0), (2.5, 0.0))            # the two inner vertices of the polyline A_19 -> ... -> B_19


def f_19(x, y):
    """3 e^{-((x - 1)² + (y - 1)²)} - 2 e^{-((x + 1)² + y²)/2} + 0.2 x (numbers or arrays)."""
    return 3 * np.exp(-((x - 1) ** 2 + (y - 1) ** 2)) - 2 * np.exp(-((x + 1) ** 2 + y ** 2) / 2) + 0.2 * x


def grad_19(p):
    """The gradient of f_19 at the point p = (x, y)."""
    x, y = p
    bump, hollow = np.exp(-((x - 1) ** 2 + (y - 1) ** 2)), np.exp(-((x + 1) ** 2 + y ** 2) / 2)
    return np.array([-6 * (x - 1) * bump + 2 * (x + 1) * hollow + 0.2, -6 * (y - 1) * bump + 2 * y * hollow])

In [ ]:
def draw_19(ax_map, ax_cum):
    """In ax_map: the level lines of f_19 on [-3, 3]², the segment from A_19 to B_19 and the polyline A_19 -> (0, -2)
    -> (2.5, 0) -> B_19, by their vertices, with a legend that names each path. In ax_cum: for each path, the integral
    of grad_19 from A_19 to the current point (line_integral on the beginning of the path) against the length travelled
    from A_19, with axis labels. Returns the two arrays of cumulative integrals drawn (the segment's, then the
    polyline's). Opens no figure."""
    raise NotImplementedError("draw_19 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB6_19a = answer_MB6_19b = ...
with wbm.attempt("MB6.19"):          # your draw_19 and line_integral: ⏳ while they are not written
    answer_MB6_19a = ...  # a) a number: f(B) - f(A)
    answer_MB6_19b = ...  # b) a list of 2 numbers, as computed: the largest cumulative integral along the segment, then along the polyline

In [ ]:
import matplotlib.contour as contour_c19

_SEGMENT_NAME_C19 = re.compile(r"segment|droit|direct", re.IGNORECASE)
_POLYLINE_NAME_C19 = re.compile(r"bris|poly|détour|detour|coud|zig", re.IGNORECASE)
PATHS_C19 = (("segment", np.array([A_19, B_19])), ("ligne brisée", np.array([A_19, *CORNERS_19, B_19])))


def distance_c19(points, vertices):
    """The distance of each point to the polyline through the vertices."""
    best = np.full(len(points), np.inf)
    for a, b in zip(vertices[:-1], vertices[1:]):
        t = np.clip(((points - a) @ (b - a)) / ((b - a) @ (b - a)), 0, 1)
        best = np.minimum(best, np.linalg.norm(points - (a + t[:, None] * (b - a)), axis=1))
    return best


def follows_c19(line, vertices):
    """True when the drawn line lies on the polyline and passes through each of its vertices."""
    points = np.column_stack([np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)])
    if points.ndim != 2 or len(points) < 2 or not np.all(np.isfinite(points)):
        return False
    return bool(distance_c19(points, vertices).max() < 1e-6
                and all(np.linalg.norm(points - v, axis=1).min() < 1e-6 for v in vertices))


def point_at_c19(vertices, length):
    """The point of the polyline at the given length from its start."""
    for k, (a, b) in enumerate(zip(vertices[:-1], vertices[1:])):
        step = float(np.linalg.norm(b - a))
        if length <= step + 1e-12 or k == len(vertices) - 2:
            return a + min(length / step, 1.0) * (b - a)
        length -= step


def cumul_why_c19(curve, vertices, values):
    """What is wrong in one curve of ax_cum (French phrases), against the integrals of the reference along the
    beginning of the path."""
    lengths = np.asarray(curve.get_xdata(), dtype=float)
    total = float(np.linalg.norm(np.diff(vertices, axis=0), axis=1).sum())
    if abs(lengths[0]) > 1e-9 or np.any(np.diff(lengths) < -1e-12) or abs(lengths[-1] - total) > 1e-6 * total:
        return ["en abscisse, la longueur parcourue depuis A_19 : de 0 à la longueur du chemin, croissante"]
    if abs(values[0]) > 1e-9:
        return ["le cumul vaut 0 au départ, en A_19"]
    ends = np.concatenate([[0.0], np.cumsum(np.linalg.norm(np.diff(vertices, axis=0), axis=1))])
    if any(np.count_nonzero((lengths >= lo - 1e-9) & (lengths <= hi + 1e-9)) < 50 for lo, hi in zip(ends, ends[1:])):
        return ["trop peu de points : il en faut au moins 50 sur chaque segment du chemin, ses deux bouts compris"]
    picks = np.unique(np.linspace(0, len(values) - 1, min(len(values), 60)).astype(int))
    # grad_19 is the gradient of f_19: its integral from A_19 to a point is f_19(point) - f_19(A_19) (fiche §4)
    expected = [float(f_19(*point_at_c19(vertices, float(lengths[k]))) - f_19(*A_19)) for k in picks]
    if np.abs(values[picks] - np.array(expected)).max() > 0.005:
        return ["les cumuls ne sont pas l'intégrale de grad_19 depuis A_19 jusqu'au point courant, à 0,005 près"]
    return []


with checked("MB6.19"):
    results_19 = []
    report_19 = reporter(results_19, stop=False)
    fig_19, (ax_map_19, ax_cum_19, spare_19) = plt.subplots(1, 3, figsize=(12, 4.5))   # spare_19: the current axes
    try:
        drawn_19 = []
        opened_19 = drawn(lambda: drawn_19.append(call_safely("Ex MB6.19 : draw_19", draw_19, ax_map_19, ax_cum_19)))
        cumuls_19 = drawn_19[0]
        arrays_19 = None
        if isinstance(cumuls_19, (list, tuple)) and len(cumuls_19) == 2:
            arrays_19 = [as_floats(c) for c in cumuls_19]
        if arrays_19 is None or any(a is None or a.ndim != 1 or a.size < 20 or not np.all(np.isfinite(a))
                                    for a in arrays_19):
            print("❌ Ex MB6.19 : draw_19 doit rendre les deux tableaux de cumuls tracés (le segment, puis la ligne "
                  f"brisée), chacun d'au moins 20 nombres finis, reçu {shown(cumuls_19)}")
            raise StopCheck
        sets_c19 = [c for c in ax_map_19.collections if isinstance(c, contour_c19.ContourSet)]
        spans_c19 = []
        for cs_c19 in sets_c19:                       # the extent of the grid given to contour (its corners)
            vertices_c19 = [p.vertices for p in cs_c19.get_paths() if len(p.vertices)]
            if vertices_c19:
                stacked_c19 = np.vstack(vertices_c19)
                lo_c19 = np.asarray(getattr(cs_c19, "_mins", stacked_c19.min(axis=0)), dtype=float)
                hi_c19 = np.asarray(getattr(cs_c19, "_maxs", stacked_c19.max(axis=0)), dtype=float)
                spans_c19.append((lo_c19, hi_c19))
        report_19(any(lo.min() >= -3.05 and hi.max() <= 3.05 and lo.max() <= -2.95 and hi.min() >= 2.95
                      for lo, hi in spans_c19),
                  "les lignes de niveau de f_19 sur [−3, 3]²",
                  "il faut les lignes de niveau de f_19 sur tout le carré [−3, 3]², par ax_map.contour (une grille "
                  "np.meshgrid de −3 à 3 sur chaque axe)")
        lines_c19 = [next((line for line in ax_map_19.lines if follows_c19(line, v)), None) for _, v in PATHS_C19]
        report_19(all(line is not None for line in lines_c19), "les deux chemins, de A_19 à B_19",
                  "il faut tracer dans ax_map le segment de A_19 à B_19 et la ligne brisée A_19 → (0, −2) → (2,5 ; 0) "
                  "→ B_19, chacun par ses sommets")
        legend_c19 = ax_map_19.get_legend()
        named_c19 = (legend_c19 is not None and lines_c19[0] is not None and lines_c19[1] is not None
                     and bool(_SEGMENT_NAME_C19.search(lines_c19[0].get_label()))
                     and bool(_POLYLINE_NAME_C19.search(lines_c19[1].get_label())))
        report_19(named_c19, "une légende qui nomme chaque chemin",
                  "il faut une légende (label=…, puis ax_map.legend()) qui nomme chaque chemin : « segment » et "
                  "« ligne brisée »")
        problems_c19 = []
        for (name_c19, vertices_c19), values_c19 in zip(PATHS_C19, arrays_19):
            curve_c19 = next((line for line in ax_cum_19.lines
                              if np.asarray(line.get_ydata(), dtype=float).shape == values_c19.shape
                              and np.allclose(np.asarray(line.get_ydata(), dtype=float), values_c19, rtol=0, atol=1e-12)),
                             None)
            if curve_c19 is None:
                problems_c19.append(f"le tableau rendu pour le {name_c19} doit être tracé dans ax_cum")
            else:
                problems_c19 += [f"{name_c19} : {why}" for why in cumul_why_c19(curve_c19, vertices_c19, values_c19)]
        if problems_c19 and not any("doit être tracé" in p for p in problems_c19):   # the two arrays swapped?
            swapped_c19 = []
            for (name_c19, vertices_c19), values_c19 in zip(PATHS_C19, arrays_19[::-1]):
                curve_c19 = next((line for line in ax_cum_19.lines
                                  if np.asarray(line.get_ydata(), dtype=float).shape == values_c19.shape
                                  and np.allclose(np.asarray(line.get_ydata(), dtype=float), values_c19, rtol=0,
                                                  atol=1e-12)), None)
                swapped_c19 += cumul_why_c19(curve_c19, vertices_c19, values_c19) if curve_c19 is not None else ["?"]
            if not swapped_c19:
                problems_c19 = ["draw_19 rend les deux tableaux dans l'autre ordre : le cumul du segment d'abord, puis "
                                "celui de la ligne brisée"]
        if not ax_cum_19.get_xlabel() or not ax_cum_19.get_ylabel():
            problems_c19.append("il faut des noms d'axes dans ax_cum")
        report_19(not problems_c19, "les deux cumuls, en fonction de la longueur parcourue, avec des noms d'axes",
                  "ax_cum : " + " ; ".join(problems_c19))
        elsewhere_19 = bool(spare_19.lines or spare_19.collections or spare_19.get_legend())
        report_19(not opened_19 and not elsewhere_19, "draw_19 dessine dans les axes reçus",
                  "draw_19 doit dessiner dans les axes reçus (ax_map.plot, ax_cum.plot…), sans ouvrir de figure ni "
                  "dessiner dans l'axe courant (plt.plot, plt.legend…)")
        fig_19.delaxes(spare_19)
    except BaseException:
        plt.close(fig_19)
        raise
    fig_19.tight_layout()
    plt.show()
    verdict("MB6.19", all(results_19) and len(results_19) == 5, "ta figure est juste : les lignes de niveau, les deux "
            "chemins, la légende et les deux cumuls.", "relis les lignes ❌ ci-dessus.")

check_answer("MB6.19a", answer_MB6_19a)
if not filled(answer_MB6_19b):
    print("⏳ Ex MB6.19b : pas encore fait.")
elif not results_19 or not all(results_19):
    print("❌ Ex MB6.19b : b) se lit sur les cumuls de draw_19 : corrige d'abord ta figure (lignes ❌ ci-dessus)")
else:
    mine_19 = as_floats(answer_MB6_19b)
    peaks_19 = [float(np.max(a)) for a in arrays_19]
    if mine_19 is None or mine_19.shape != (2,):
        print(f"❌ Ex MB6.19b : une liste de 2 nombres, reçu {shown(answer_MB6_19b)}")
    elif not np.allclose(mine_19, peaks_19, rtol=1e-9, atol=1e-12):
        print("❌ Ex MB6.19b : b) relève le plus grand cumul de chacun des deux tableaux que rend draw_19 (le segment, "
              "puis la ligne brisée), sans l'arrondir")
    elif not passes("MB6.19b", peaks_19):
        print("❌ Ex MB6.19b : tes plus grands cumuls sont loin de ceux des deux chemins : répartis tes points sur toute "
              "la longueur de chaque segment (np.linspace), pour ne pas manquer le haut de la bosse")
    else:
        print("✅ Ex MB6.19b : les plus grands cumuls de tes deux tableaux, assez denses pour trouver les sommets.")

**Ce qu'il faut regarder** : où chaque chemin monte-t-il et descend-il ? Pourquoi les deux cumuls finissent-ils au même nombre ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie C · Déboguer, tester

Fiche §1, §4, §5 et §6. Trois erreurs, qui passent une vérification mal choisie ; trois tests de propriété d'une intégrale le long d'une ligne brisée. Les méthodes cachées s'appuient sur la référence (`ref_att`), pour ne pas dépendre de ton code.

### Ex MB6.20 — Chasse à l'erreur : des poids de coalition sans le facteur 1/n, des ex aequo classés dans l'ordre d'apparition, une interpolation linéaire aux poids échangés 🐛 ★ ⏱️ 20 min
**Objectif :** reconnaître trois erreurs (des poids sans le facteur $\frac1n$, des ex aequo rangés dans l'ordre d'apparition, une interpolation aux poids échangés), qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §1, §5 et §6 ; Ex MB6.13, MB6.16 · **Parcours :** C, E

Un collègue a écrit trois fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `weights_bad(n)` doit rendre les poids $w(s)$ de la valeur de Shapley ; les poids, comptés autant de fois qu'il y a de coalitions de chaque taille, ne somment pas à 1. Il l'a vérifiée pour $n = 1$.
2. `ranks_bad(x)` doit rendre les rangs, rangs moyens aux ex aequo ; des valeurs égales n'ont pas le même rang. Il l'a vérifiée sur un échantillon sans ex aequo.
3. `lerp_bad(a, b, t)` doit interpoler entre $a$ (en $t = 0$) et $b$ (en $t = 1$) ; la valeur lue est du mauvais côté. Elle sert à lire une valeur entre deux cases d'une grille (fiche §5). Il l'a vérifiée en $t = \frac12$.

À faire : trouve et corrige chaque erreur dans le code du collègue lui-même : les fonctions corrigées s'appellent `weights_fixed`, `ranks_fixed` et `lerp_fixed`, avec les mêmes arguments, et n'appellent **pas** `mymath` (la vérification le contrôle). Puis réponds :
- a) pour $n = 7$, $\sum_s\binom{6}{s}w(s)$ avec la fonction corrigée, puis avec la fonction fautive (une liste de 2 nombres) ;
- b) $\rho_S$ des deux échantillons `X_20` et `Y_20` (avec des ex aequo), calculée par la corrélation de Pearson des rangs de la fonction corrigée, puis de ceux de la fonction fautive (une liste de 2 nombres ; hors de `mymath`, `np.corrcoef` est permis) ;
- c) l'interpolation en $t = 0{,}3$ entre 12 et 20, corrigée puis fautive (une liste de 2 nombres).

Dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie aussi tes fonctions corrigées sur des cas cachés : d'autres nombres de joueurs, d'autres échantillons avec des ex aequo, d'autres valeurs lues entre deux cases, et des tableaux qu'elles ne doivent pas modifier.

In [ ]:
# The colleague's code
def weights_bad(n):
    """The weights w(s) of the Shapley value of n players, s = 0, ..., n - 1 (floats)."""
    return [math.factorial(s) * math.factorial(n - 1 - s) / math.factorial(n - 1) for s in range(n)]


def ranks_bad(x):
    """The ranks of the values of x, from 1 (the smallest), tied values sharing the mean of the places they occupy."""
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="stable")
    ranks = np.empty(len(x))
    ranks[order] = np.arange(1, len(x) + 1)
    return ranks


def lerp_bad(a, b, t):
    """The value at t of the linear interpolation between a (at t = 0) and b (at t = 1), to read a value between two
    cells of a grid."""
    return (1 - t) * b + t * a


X_20 = frozen([5.1, 3.5, 4.3, 5.1, 4.7, 4.7, 4.7])                # two samples with ties (b)
Y_20 = frozen([3.0, 1.0, 2.0, 4.5, 5.0, 1.5, 4.0])

# The colleague's checks: all pass
print("(1)", weights_bad(1) == [1.0], "(attendu True : un joueur seul reçoit tout)")
print("(2)", ranks_bad([2.7, 0.4, 1.9, 3.6]).tolist() == [3.0, 1.0, 2.0, 4.0], "(attendu True : des valeurs toutes "
      "différentes)")
print("(3)", lerp_bad(4.0, 10.0, 0.5) == 7.0, "(attendu True : au milieu, la moyenne des deux valeurs)")

In [ ]:
def weights_fixed(n):
    """The weights w(s) of the Shapley value of n players, s = 0, ..., n - 1 (floats)."""
    raise NotImplementedError("weights_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def ranks_fixed(x):
    """The ranks of the values of x, from 1 (the smallest), tied values sharing the mean of the places they occupy."""
    raise NotImplementedError("ranks_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def lerp_fixed(a, b, t):
    """The value at t of the linear interpolation between a (at t = 0) and b (at t = 1), to read a value between two
    cells of a grid."""
    raise NotImplementedError("lerp_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB6_20a = answer_MB6_20b = answer_MB6_20c = ...
with wbm.attempt("MB6.20"):          # your three functions: ⏳ while they are not written
    answer_MB6_20a = ...  # a) a list of 2 numbers, as computed: for n = 7, sum of C(6, s) w(s) by weights_fixed, then by the WRONG weights_bad
    answer_MB6_20b = ...  # b) a list of 2 numbers, as computed: rho_S of X_20, Y_20 on the ranks of ranks_fixed, then of the WRONG ranks_bad
    answer_MB6_20c = ...  # c) a list of 2 numbers, as computed: at t = 0.3 between 12 and 20, by lerp_fixed, then by the WRONG lerp_bad

In [ ]:
for letter, answer in zip("abc", [answer_MB6_20a, answer_MB6_20b, answer_MB6_20c]):
    check_answer(f"MB6.20{letter}", answer)

CASES_C20 = {'players': [11, 14, 17],
 'samples': [[3.71, 1.94, 3.71, 2.58, 1.94, 3.71, 0.66], [-0.37, 2.15, 2.15, -1.08, 0.94, 2.15, 0.94, 5.26],
             [7.3, 6.1, 6.1, 8.9, 7.3, 6.1, 5.2, 9.4, 7.3]],
 'lerps': [(-3.7, 5.9, 0.23), (12.4, 2.3, 0.85), (0.61, 8.43, 0.38),
           ([1.37, -0.64, 2.91], [4.03, 0.29, -1.58], 0.35)],
 'id_weights': 'MB6.20d',
 'id_ranks': 'MB6.20e',
 'id_lerps': 'MB6.20f'}   # the hidden cases, and the sub-IDs of the values expected

results_20, parts_20 = [], []
MYMATH_C20 = list(getattr(mymath, "__path__", [])) + list(getattr(REF, "__path__", []))   # the folders of mymath
TWINS_C20 = set()                                  # the functions of attribution.py (yours, the reference's)
for _folder_c20 in MYMATH_C20:
    TWINS_C20 |= code_keys_of_file(os.path.join(_folder_c20, "attribution.py"))


def report_20(ok, success, failure):
    results_20.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_20(part, name, function, *args):
    """function(*args), without mymath; an error of the learner's code becomes one ❌ line that names the function."""
    try:
        with code_from(MYMATH_C20, twins=TWINS_C20) as seen:
            got = function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_20(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:160]}")
        parts_20.append(part)
        raise StopCheck from None
    if seen:
        report_20(False, "", f"({part}) {name} appelle mymath ({seen[0]}) : l'énoncé demande la correction sans mymath, "
                  "en corrigeant le code du collègue")
        parts_20.append(part)
        raise StopCheck
    return got


with checked("MB6.20"):                                       # (1) weights_fixed
    got_c20, bad_c20, shapes_c20 = [], [], True
    for n_c20 in CASES_C20["players"]:
        weights_c20 = as_floats(call_20(1, "weights_fixed", weights_fixed, n_c20))
        shapes_c20 = shapes_c20 and weights_c20 is not None and weights_c20.shape == (n_c20,)
        got_c20 += weights_c20.tolist() if shapes_c20 else []
        bad_c20 += weights_bad(n_c20)
    if not shapes_c20:
        why_c20 = "weights_fixed(n) rend les n poids w(0), …, w(n − 1)"
    elif passes(CASES_C20["id_weights"], got_c20):
        why_c20 = ""
    elif np.allclose(got_c20, bad_c20, rtol=1e-9, atol=0):
        why_c20 = "ta correction divise encore s! (n − 1 − s)! par (n − 1)!"
    else:
        why_c20 = "pour d'autres nombres de joueurs, ce ne sont pas les poids w(s) de la valeur de Shapley"
    report_20(not why_c20, "(1) weights_fixed : d'autres nombres de joueurs", f"(1) weights_fixed : {why_c20}")
    parts_20.append(1)

with checked("MB6.20"):                                       # (2) ranks_fixed
    got_c20, bad_c20, why_c20 = [], [], ""
    for sample_c20 in CASES_C20["samples"]:
        x_c20 = np.array(sample_c20)                          # non-dyadic floats: a change in place would show
        ranks_c20 = as_floats(call_20(2, "ranks_fixed", ranks_fixed, x_c20))
        if not np.array_equal(x_c20, np.array(sample_c20)):
            why_c20 = "ta fonction modifie l'échantillon reçu : travaille sur une copie"
            break
        if ranks_c20 is None or ranks_c20.shape != x_c20.shape:
            why_c20 = "ranks_fixed rend un rang par valeur, de la forme de x"
            break
        got_c20 += ranks_c20.tolist()
        bad_c20 += ranks_bad(sample_c20).tolist()
    if not why_c20 and not passes(CASES_C20["id_ranks"], got_c20):
        why_c20 = ("ta correction donne encore aux ex aequo des rangs différents" if got_c20 == bad_c20 else
                   "sur d'autres échantillons (des groupes de 2 et de 3 valeurs égales), ce ne sont pas les rangs "
                   "moyens : chaque groupe de valeurs égales reçoit la moyenne des places qu'il occupe")
    report_20(not why_c20, "(2) ranks_fixed : d'autres échantillons, avec des groupes d'ex aequo de tailles différentes",
              f"(2) ranks_fixed : {why_c20}")
    parts_20.append(2)

with checked("MB6.20"):                                       # (3) lerp_fixed, read directly
    got_c20, bad_c20, why_c20 = [], [], ""
    for a_c20, b_c20, t_c20 in CASES_C20["lerps"]:
        left_c20, right_c20 = np.array(a_c20), np.array(b_c20)   # non-dyadic floats: a change in place would show
        value_c20 = as_floats(call_20(3, "lerp_fixed", lerp_fixed, left_c20, right_c20, t_c20))
        if not (np.array_equal(left_c20, np.array(a_c20)) and np.array_equal(right_c20, np.array(b_c20))):
            why_c20 = "ta fonction modifie un tableau reçu : travaille sur une copie"
            break
        if value_c20 is None or value_c20.shape != np.shape(a_c20):
            why_c20 = "lerp_fixed rend une valeur de la forme de a"
            break
        got_c20 += np.ravel(value_c20).tolist()
        bad_c20 += np.ravel(lerp_bad(np.array(a_c20), np.array(b_c20), t_c20)).tolist()
    if not why_c20 and not passes(CASES_C20["id_lerps"], got_c20):
        why_c20 = ("ta correction rend encore la valeur b en t = 0" if np.allclose(got_c20, bad_c20, rtol=1e-12) else
                   "en d'autres t (ni 0, ni ½, ni 1), ce n'est pas l'interpolation qui vaut a en t = 0 et b en t = 1")
    report_20(not why_c20, "(3) lerp_fixed : d'autres valeurs lues entre deux cases, en nombres et en tableaux",
              f"(3) lerp_fixed : {why_c20}")
    parts_20.append(3)

if len(set(parts_20)) == 3:
    verdict("MB6.20", all(results_20), "les trois fonctions corrigées passent tous les cas.", "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait-elle pas, et quelle vérification l'aurait vue ?

### Ex MB6.21 — Un test de propriété sur des lignes brisées tirées au hasard : le chemin inversé change le signe, un segment coupé en deux ne change rien, un champ de gradient donne f(fin) − f(début) 🛠️ ★ ⏱️ 25 min
**Objectif :** écrire trois tests de propriété d'une intégrale le long d'une ligne brisée, et voir ce que chacun attrape.  
**Prérequis :** fiche §4 ; M02a (méthode de Simpson) ; Ex MB6.15 · **Parcours :** C

Écris trois fonctions qui testent une méthode qu'on leur passe, de la même signature que `line_integral(field, points)`, et rendent `True` si la propriété tient, `False` sinon :
- `reversed_changes_sign_21(method)` vérifie, sur plusieurs lignes brisées tirées au hasard dans le plan et un champ qui n'est pas un gradient, que parcourir la ligne à l'envers change le signe ;
- `split_changes_nothing_21(method)` vérifie qu'ajouter un sommet à l'intérieur d'un segment (en un point tiré au hasard sur ce segment) ne change pas le résultat ;
- `gradient_gives_difference_21(method)` vérifie, pour le gradient d'une fonction dont tu connais l'expression, que le résultat vaut $f(\text{fin}) - f(\text{début})$.

Chaque comparaison se fait à $10^{-9}$ près en valeur relative, plus finement que la tolérance par défaut de `np.allclose` (`rtol=1e-5`, `atol=1e-8`). Choisis tes champs pour que chaque propriété tienne à l'arrondi près pour `line_integral` de la référence (`ref_att.line_integral`) : parcourir un segment à l'envers garde les mêmes points de Simpson, pour tout champ, mais une méthode aux points non symétriques ne se voit que si $\mathbf{F}\cdot\boldsymbol\gamma'$ varie le long d'un segment (ce n'est pas le cas du champ tournant $(-y, x)$) ; la règle de Simpson n'est exacte que si $t \mapsto \mathbf{F}(\boldsymbol\gamma(t))\cdot\boldsymbol\gamma'(t)$ est un polynôme de degré au plus 3 sur chaque segment, et, pour un autre champ, couper un segment en deux change le résultat d'environ $10^{-5}$ : pour les deux derniers tests, prends des champs dont les coordonnées sont des polynômes de degré 2 ou 3.

La vérification lance tes trois tests sur `line_integral` de la référence, sur deux autres méthodes justes, puis sur des méthodes fausses cachées, passées sous un nom neutre (un test juge ce que fait la méthode, pas son nom), une par clause de l'énoncé, que chaque test doit refuser quand sa propriété est fausse. Un test exécute la méthode sur ses lignes brisées et compare ses résultats à ce que dit la propriété : il ne lit ni la méthode ni la pile d'appels, et n'appelle pas `line_integral` de la référence (la vérification le contrôle). Relève :
- a) les résultats de tes trois tests sur `line_integral` de la référence (une liste de 3 booléens, dans l'ordre sens inversé, segment coupé, gradient).

In [ ]:
def reversed_changes_sign_21(method):
    """True when, on several polylines drawn at random and a field that is not a gradient, method(field, points[::-1])
    is -method(field, points) up to 1e-9 in relative value; False otherwise."""
    raise NotImplementedError("reversed_changes_sign_21 n'est pas encore écrite (04_indices.md si tu bloques)")


def split_changes_nothing_21(method):
    """True when adding a vertex inside a segment (at a point drawn at random on it) does not change method(field,
    points), up to 1e-9 in relative value, on several polylines drawn at random; False otherwise."""
    raise NotImplementedError("split_changes_nothing_21 n'est pas encore écrite (04_indices.md si tu bloques)")


def gradient_gives_difference_21(method):
    """True when, for the gradient of a function f you know, method(gradient, points) is f(last point) - f(first
    point) up to 1e-9 in relative value, on several polylines drawn at random; False otherwise."""
    raise NotImplementedError("gradient_gives_difference_21 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB6_21a = ...
with wbm.attempt("MB6.21"):          # your three tests: ⏳ while they are not written
    answer_MB6_21a = ...  # a) a list of 3 booleans: your three tests on ref_att.line_integral

In [ ]:
for letter, answer in zip("a", [answer_MB6_21a]):
    check_answer(f"MB6.21{letter}", answer)

def load_hidden_c21():
    """The quadratures of the check (solutions/challenges/mb6_21.py: reading it spoils 🔨 MB6.15)."""
    import importlib.util
    spec = importlib.util.spec_from_file_location("hidden_methods_c21", wbm.repo_root() / "solutions" / "challenges"
                                                  / "mb6_21.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


HIDDEN_C21 = load_hidden_c21()
simpson_twin_c21, gauss_twin_c21 = HIDDEN_C21.simpson_twin, HIDDEN_C21.gauss_twin
left_sums_c21, trapezoid_c21, positive_steps_c21 = HIDDEN_C21.left_sums, HIDDEN_C21.trapezoid, HIDDEN_C21.positive_steps


def absolute_c21(field, points):
    """WRONG: the absolute value of the integral (a path backwards keeps its sign)."""
    return abs(ref_att.line_integral(field, points))


def first_segment_c21(field, points):
    """WRONG from two segments on: the integral along the first segment only."""
    return ref_att.line_integral(field, np.asarray(points, dtype=float)[:2])


def swapped_c21(field, points):
    """WRONG for a field whose components differ: F₂ dx + F₁ dy instead of F₁ dx + F₂ dy."""
    return ref_att.line_integral(lambda p: np.asarray(field(p), dtype=float)[::-1], points)


def plus_offset_c21(field, points):
    """WRONG by more than 1e-9, but within a tolerance of 1e-7 and np.allclose's default (1e-8 + 1e-5 |x|): the
    integral plus 2e-9 + 2e-8 |integral|."""
    value = ref_att.line_integral(field, points)
    return value + 2e-9 + 2e-8 * abs(value)


def per_vertex_c21(field, points):
    """WRONG by more than 1e-9, but within a tolerance of 1e-7 and np.allclose's default: the integral times
    1 + 2e-8 (k - 2), k the number of vertices (one vertex more changes the result)."""
    value = ref_att.line_integral(field, points)
    return value * (1 + 2e-8 * (len(points) - 2))


WRONG_C21 = [   # a hidden wrong method, the test that must refuse it, and the clause of the statement it tries
    (left_sums_c21, "reversed_changes_sign_21", "dès qu'on parcourt le chemin à l'envers (ses points d'évaluation ne "
                                                "sont pas symétriques) : ton champ fait-il varier F · γ′ le long d'un "
                                                "segment ? Le champ tournant (−y, x) le garde constant"),
    (absolute_c21, "reversed_changes_sign_21", "sur le signe : elle rend la même valeur dans les deux sens"),
    (first_segment_c21, "reversed_changes_sign_21", "dès deux segments (un seul segment est juste) : tes lignes brisées "
                                                    "en ont-elles plusieurs ?"),
    (plus_offset_c21, "reversed_changes_sign_21", "de 2·10⁻⁹ + 2·10⁻⁸ |valeur| : la tolérance de l'énoncé est 10⁻⁹ en "
                                                  "valeur relative (ta tolérance est-elle plus large ? np.allclose a par "
                                                  "défaut rtol = 1e-5 et atol = 1e-8)"),
    (trapezoid_c21, "split_changes_nothing_21", "sauf pour un champ affine (la méthode des trapèzes) : prends un champ "
                                                "dont les coordonnées sont de degré 2 ou 3"),
    (per_vertex_c21, "split_changes_nothing_21", "d'un facteur 1 + 2·10⁻⁸ par sommet ajouté : la tolérance de l'énoncé "
                                                 "est 10⁻⁹ en valeur relative (ta tolérance est-elle plus large ? "
                                                 "np.allclose a par défaut rtol = 1e-5 et atol = 1e-8)"),
    (trapezoid_c21, "gradient_gives_difference_21", "sauf pour un champ affine (la méthode des trapèzes) : prends le "
                                                    "gradient d'un polynôme de degré 3"),
    (first_segment_c21, "gradient_gives_difference_21", "dès deux segments (un seul segment est juste) : tes lignes "
                                                        "brisées en ont-elles plusieurs ?"),
    (swapped_c21, "gradient_gives_difference_21", "pour un gradient dont les deux coordonnées diffèrent (elle échange "
                                                  "les composantes du champ) : prends une fonction dont les deux "
                                                  "dérivées partielles diffèrent, comme x³y"),
    (plus_offset_c21, "gradient_gives_difference_21", "de 2·10⁻⁹ + 2·10⁻⁸ |valeur| : la tolérance de l'énoncé est 10⁻⁹ "
                                                      "en valeur relative (ta tolérance est-elle plus large ? np.allclose "
                                                      "a par défaut rtol = 1e-5 et atol = 1e-8)"),
    (positive_steps_c21, "gradient_gives_difference_21", "dès qu'un segment va vers la gauche ou vers le bas (elle prend "
                                                         "|b − a| pour b − a) : tes lignes brisées, tirées au hasard, "
                                                         "vont-elles dans toutes les directions ?"),
]

with checked("MB6.21"):
    results_21 = []
    report_21 = reporter(results_21, stop=False)
    TESTS_C21 = {"reversed_changes_sign_21": reversed_changes_sign_21, "split_changes_nothing_21": split_changes_nothing_21,
                 "gradient_gives_difference_21": gradient_gives_difference_21}

    def anonymous_c21(method):
        """method under a neutral name: a test must judge what the method does, not what it is called."""
        def method_c21(field, points):
            return method(field, points)
        method_c21.__name__ = method_c21.__qualname__ = "method"
        return method_c21

    def verdict_c21(name, label, method):
        got = call_safely(f"Ex MB6.21 : {name}({label})", TESTS_C21[name], anonymous_c21(method))
        if not isinstance(got, (bool, np.bool_)):
            if isinstance(got, (int, np.integer)):
                what = f"{int(got)} (un entier)"
            elif isinstance(got, (float, np.floating)):
                what = f"{float(got):g} (un nombre)"
            else:
                what = shown(got)
            print(f"❌ Ex MB6.21 : {name}({label}) rend {what}, pas True ou False")
            raise StopCheck
        return bool(got)

    def verdicts_c21(label, method):
        return [verdict_c21(name, label, method) for name in TESTS_C21]

    # a test runs the method: it reads neither the method, nor the stack, nor the names of this cell
    refuse_peeking("MB6.21", TESTS_C21, globals(), private=("_c21", "_C21"))

    REF_FILE_C21 = REF.attribution.__file__

    def no_reference_c21(label, method):
        """The verdicts on a method that never calls the reference: a call of the reference comes from the test."""
        with code_from([REF_FILE_C21]) as seen:
            got = verdicts_c21(label, method)
        if seen:
            print(f"❌ Ex MB6.21 : un de tes tests appelle line_integral de la référence ({seen[0]}) : il doit comparer "
                  "la méthode à −valeur, à la valeur sans le sommet ajouté, ou à f(fin) − f(début), pas à la référence")
            raise StopCheck
        return got

    on_ref_c21 = verdicts_c21("line_integral de la référence", ref_att.line_integral)
    report_21(passes("MB6.21a", on_ref_c21), "tes trois tests, sur line_integral de la référence",
              "tes trois tests ne rendent pas True sur line_integral de la référence, qui vérifie les trois propriétés : "
              "relis le choix des champs (fiche §4, et la règle de Simpson, exacte pour un polynôme de degré au plus 3)")
    twins_c21 = [no_reference_c21("une autre écriture juste de Simpson", simpson_twin_c21),
                 no_reference_c21("une règle de Gauss juste", gauss_twin_c21)]
    report_21(all(all(t) for t in twins_c21), "deux autres méthodes justes reçoivent aussi True",
              "deux autres méthodes justes (Simpson à 40 intervalles écrite autrement, Gauss à trois points ; des "
              "arrondis de l'ordre de 10⁻¹⁵) ne reçoivent pas toutes True : ta tolérance est-elle trop stricte, ou un "
              "champ trop éloigné d'un polynôme de degré au plus 3 ?")
    for number_c21, (method_c21, name_c21, clause_c21) in enumerate(WRONG_C21, start=1):
        refused_c21 = not verdict_c21(name_c21, f"méthode fausse cachée n° {number_c21}", method_c21)
        report_21(refused_c21, f"{name_c21} refuse la méthode fausse cachée n° {number_c21}",
                  f"{name_c21} accepte la méthode fausse cachée n° {number_c21}, fausse {clause_c21}")
    verdict("MB6.21", all(results_21) and len(results_21) == 2 + len(WRONG_C21), "tes trois tests jugent juste "
            "line_integral de la référence et deux autres méthodes justes, et refusent les méthodes fausses cachées.",
            "relis les lignes ❌ ci-dessus.")

**Ce qu'il faut regarder** : pourquoi un champ non polynomial ferait-il refuser `line_integral` de la référence par ton deuxième test ? Pourquoi un champ affine laisserait-il passer la méthode des trapèzes ? Pourquoi tirer les lignes brisées au hasard, et plusieurs fois ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie D · Défi

Fiche §1. Les indices de Shapley-Shubik exacts d'un conseil de 25 membres, par un comptage par taille et par poids, vérifiés sur des conseils cachés tirés à chaque exécution.

### Ex MB6.22 — Défi : les indices de Shapley-Shubik exacts d'un conseil de 25 membres, sans parcourir ses coalitions une à une, en comptant par taille et par poids celles où chaque membre est pivot 🏆 ★★★ ⏱️ 45 min
**Objectif :** calculer les indices de Shapley-Shubik exacts d'un grand conseil par un comptage, et le vérifier sur des conseils cachés.  
**Prérequis :** fiche §1 ; Ex MB6.1, MB6.13 · **Parcours :** C

Le notebook fournit `WEIGHTS_22`, les voix des 25 membres d'un conseil (des entiers de 1 à 12, 98 voix en tout), et `QUOTA_22` $= 66$. Écris `shapley_shubik_22(weights, quota)`, qui rend les indices de Shapley-Shubik de tous les membres, exacts (des flottants), sans parcourir les coalitions une à une : pour chaque membre $i$, compte les coalitions des autres membres par taille et par poids total (un tableau rempli membre par membre), garde celles où $i$ est pivot, et pondère par `coalition_weights`.

**Critère de réussite :** pour le conseil du notebook et pour trois séries de cinq conseils cachés de 30 à 50 membres, tirés à nouveau à chaque exécution (une majorité simple, une majorité qualifiée, un membre au veto, quelques grands et beaucoup de petits, des poids presque égaux), chaque indice est à $10^{-9}$ près l'indice exact, et ta fonction traite les quinze conseils cachés en moins de cinq minutes. Elle ne doit ni exécuter le code du module caché du défi (`solutions/challenges/mb6_22.py`), ni lire la vérification (sa pile d'appels, le processus à part) : la vérification le contrôle. Relève :
- a) l'indice du membre 0, qui a 12 voix (un nombre) ;
- b) les 25 indices (une liste de 25 nombres).

Point de départ : pour un conseil de 30 membres, les coalitions sans un membre sont $2^{29}$, plus de 500 millions ; mais leurs poids totaux sont des entiers de 0 à quelques centaines.

In [ ]:
# The council of the challenge: the votes of its 25 members (0 to 24), and the quota
WEIGHTS_22 = (12, 9, 9, 7, 6, 6, 5, 5, 4, 4, 4, 3, 3, 3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1)
QUOTA_22 = 66

In [ ]:
def shapley_shubik_22(weights, quota):
    """The Shapley-Shubik indices of all the members of the council [quota ; weights] (positive integers), exact
    (floats), without going through the coalitions one by one."""
    raise NotImplementedError("shapley_shubik_22 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB6_22a = answer_MB6_22b = ...
with wbm.attempt("MB6.22"):          # your shapley_shubik_22: ⏳ while it is not written
    answer_MB6_22a = ...  # a) a number, as computed: the index of member 0 (12 votes)
    answer_MB6_22b = ...  # b) a list of 25 numbers, as computed: the 25 indices

In [ ]:
for letter, answer in zip("ab", [answer_MB6_22a, answer_MB6_22b]):
    check_answer(f"MB6.22{letter}", answer)


def check_c22():
    """The check of MB6.22, in a function. The hidden councils come from another process (the hidden module run as a
    script): it draws them, sends their weights and quotas, and measures the indices it receives only at the end;
    no exact index of a hidden council lives in this process while the learner's function runs. Each call runs under
    code_from (the learner's function may not run code of the hidden module) and under a time limit where the system
    allows one (it only stops a runaway computation)."""
    import json
    import signal
    import subprocess
    import sys
    import threading
    refuse_peeking("MB6.22", {"shapley_shubik_22": shapley_shubik_22}, globals(),
                   private=("_c22", "_C22"))            # it computes, it never reads the check
    hidden_file = wbm.repo_root() / "solutions" / "challenges" / "mb6_22.py"   # reading it spoils the challenge
    twins = code_keys_of_file(hidden_file)                      # its functions, to recognise a copy run from elsewhere
    results = []
    report = reporter(results, stop=False)
    budget = 300.0                                              # seconds for the fifteen hidden councils

    class TimeUp(BaseException):
        """Raised by the alarm: a BaseException, so that a broad except of the learner's code does not hide it."""

    @contextlib.contextmanager
    def time_limit(seconds):
        usable = hasattr(signal, "setitimer") and threading.current_thread() is threading.main_thread()
        if not usable:
            yield
            return

        def ring(signum, frame):
            raise TimeUp

        before = signal.signal(signal.SIGALRM, ring)
        signal.setitimer(signal.ITIMER_REAL, max(seconds, 0.01))
        try:
            yield
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, before)

    def indices(label, weights, quota, seconds):
        """shapley_shubik_22 on a council, under code_from and a time limit: (a float array of one index per member,
        the time taken)."""
        start = time.perf_counter()
        try:
            with time_limit(seconds):
                with code_from([hidden_file], twins=twins, escapes=True) as seen:
                    out = call_safely(label, shapley_shubik_22, list(weights), int(quota))
        except TimeUp:
            print(f"❌ {label} : ta fonction n'a pas fini en {fr(seconds, 0)} s : parcourt-elle les coalitions une à une ? "
                  "Compte-les par taille et par poids total")
            raise StopCheck from None
        elapsed = time.perf_counter() - start
        if seen and seen[0].startswith("lance "):
            print(f"❌ {label} : shapley_shubik_22 {seen[0].replace(' : ', ' (', 1)}) : les indices se calculent dans ta "
                  "fonction ; ce qu'exécute un autre processus échappe à la vérification")
            raise StopCheck
        if seen:
            what = "lit" if seen[0].startswith("ouvre ") else "exécute du code de"
            where = "" if seen[0].startswith("ouvre ") else f" ({seen[0]})"
            print(f"❌ {label} : shapley_shubik_22 {what} solutions/challenges/mb6_22.py{where}, la partie cachée du "
                  "défi, ou une copie de son code : la vérification mesurerait sa méthode, pas la tienne")
            raise StopCheck
        values = as_floats(out)
        if values is None or values.shape != (len(weights),):
            print(f"❌ {label} : shapley_shubik_22 rend un indice par membre du conseil reçu ({len(weights)} nombres), reçu "
                  f"{shown(out)}")
            raise StopCheck
        if not np.all(np.isfinite(values)):
            print(f"❌ {label} : shapley_shubik_22 rend des nombres qui ne sont pas tous finis")
            raise StopCheck
        return values, elapsed

    def probe(n):
        """A council of n members of 1, 2, ..., 8, 1, 2, ... votes, simple majority (a probe of the time, not a hidden
        case): the smaller of two timings."""
        votes = [k % 8 + 1 for k in range(n)]
        label = f"Ex MB6.22 : shapley_shubik_22 (un conseil d'essai de {n} membres)"
        return min(indices(label, votes, sum(votes) // 2 + 1, 60)[1] for _ in range(2))

    mine, spent = indices("Ex MB6.22 : shapley_shubik_22 (le conseil du notebook)", WEIGHTS_22, QUOTA_22, 30)
    if spent > 30:
        print(f"❌ Ex MB6.22 : ta fonction met {fr(spent, 1)} s pour les 25 membres du notebook ; les conseils cachés en "
              "ont 30 à 50 : à ce rythme, ils dépasseraient de loin les cinq minutes permises")
        raise StopCheck
    if spent >= 0.2:                                            # a slow function: does its time double with each member?
        times = [probe(n) for n in (16, 22)]                    # 2^6 = 64 times more coalitions; a polynomial: 3 to 5 times
        if times[1] > 0.05 and times[1] > 15 * max(times[0], 1e-3):
            print(f"❌ Ex MB6.22 : ta fonction semble parcourir les coalitions : son temps est multiplié par "
                  f"{fr(times[1] / max(times[0], 1e-3), 0)} quand le conseil passe de 16 à 22 membres ({fr(times[0], 3)} s, "
                  f"puis {fr(times[1], 2)} s) ; pour 30 à 50 membres, il lui faudrait des heures : compte les coalitions "
                  "par taille et par poids total")
            raise StopCheck

    def exactly_line(stream):
        line = stream.readline()
        if not line.endswith(b"\n"):
            raise OSError("échange interrompu")
        return json.loads(line)

    server = subprocess.Popen([sys.executable, "-I", str(hidden_file), "--serve"], stdin=subprocess.PIPE,
                              stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)   # this kernel's Python, here or on Colab
    try:
        councils = exactly_line(server.stdout)["councils"]
        answers, total = [], 0.0
        for k, (weights, quota) in enumerate(councils, start=1):
            values, spent = indices(f"Ex MB6.22 : shapley_shubik_22 (conseil caché {k})", weights, quota,
                                    max(budget - total, 1.0))
            answers.append(values.tolist())
            total += spent
        server.stdin.write((json.dumps(answers + [mine.tolist()]) + "\n").encode())
        server.stdin.flush()
        measured = exactly_line(server.stdout)                    # only now: the gaps, the kinds
        assert len(measured) == len(councils) + 1
    except (OSError, ValueError, KeyError, TypeError, AssertionError) as exc:
        print("❌ Ex MB6.22 : la vérification n'a pas pu échanger les conseils et les indices avec le processus qui tire "
              f"les conseils cachés ({type(exc).__name__} ; ta fonction a-t-elle touché à ce processus ?) : exécute à "
              "nouveau la cellule")
        raise StopCheck from None
    finally:                                                    # always, even on an error or an interruption
        if server.poll() is None:
            server.kill()
        server.wait()
        for pipe in (server.stdin, server.stdout):
            try:
                pipe.close()
            except OSError:                                     # a pipe already broken
                pass

    def tens(x):
        """A gap written the French way, in powers of ten (9,0·10⁻³)."""
        if x == 0:
            return "0"
        mantissa, exponent = f"{x:.1e}".split("e")
        if int(exponent) == 0:
            return mantissa.replace(".", ",")
        return mantissa.replace(".", ",") + "·10" + str(int(exponent)).translate(str.maketrans("-0123456789",
                                                                                                "⁻⁰¹²³⁴⁵⁶⁷⁸⁹"))

    notebook, hidden = measured[-1], measured[:-1]
    notebook_ok = notebook["gap"] is not None and notebook["gap"] <= 1e-9
    report(notebook_ok, "le conseil du notebook : chaque indice à 10⁻⁹ près",
           "le conseil du notebook : " + ("des indices qui ne sont pas des nombres finis" if notebook["gap"] is None
                                          else f"un indice à {tens(notebook['gap'])} de l'indice exact, plus que 10⁻⁹"))
    for number in sorted({c["series"] for c in hidden}):        # what each hidden council measured (no index)
        series = [c for c in hidden if c["series"] == number]
        print(f"   série {number} : " + " ; ".join(f"« {c['kind']} » ({c['n']} membres) écart "
                                                  + ("non mesuré" if c["gap"] is None else tens(c["gap"]))
                                                  for c in series))
    wrong = [c for c in hidden if c["gap"] is None or c["gap"] > 1e-9]
    largest = max((c["gap"] for c in wrong if c["gap"] is not None), default=None)
    if not wrong:
        why = ""
    elif largest is not None and largest < 1e-6 and all(c["gap"] is not None for c in wrong):
        why = (f" (écart jusqu'à {tens(largest)}) : presque juste, mais les comptes de coalitions vont jusqu'à 10¹⁴ ; "
               "des flottants de 32 bits n'en gardent pas assez de chiffres : garde des flottants de 64 bits ou des "
               "entiers")
    elif notebook_ok:
        why = (" : ta fonction est exacte pour le conseil du notebook, mais pas pour ces conseils ; se sert-elle de "
               "nombres du conseil du notebook (25 membres, le quota 66, ses voix) au lieu de ceux du conseil reçu ?")
    else:
        why = ("" if largest is None else f" (écart jusqu'à {tens(largest)})") + (" : ce ne sont pas les indices de "
               "Shapley-Shubik ; relis la condition du pivot et les poids w(s) des coalitions (fiche §1)")
    report(not wrong, f"les {len(hidden)} conseils cachés : chaque indice à 10⁻⁹ près, en {fr(total, 2)} s",
           f"les {len(hidden)} conseils cachés : {len(wrong)} ont un indice à plus de 10⁻⁹ de l'indice exact" + why)
    report(total <= budget, "le temps : moins de cinq minutes pour les quinze conseils cachés",
           f"le temps : {fr(total, 1)} s pour les quinze conseils cachés, plus que les cinq minutes permises")
    verdict("MB6.22", all(results) and len(results) == 3, "critère atteint : les indices exacts du conseil du notebook "
            "et des quinze conseils cachés, à 10⁻⁹ près, en moins de cinq minutes.",
            "critère pas encore atteint : relis les lignes ❌ ci-dessus.")


with checked("MB6.22"):
    check_c22()

**Pour réfléchir** (une fois le critère atteint) : pourquoi deux membres de mêmes voix ont-ils le même indice ? Le membre de 12 voix a-t-il un pouvoir proportionnel à ses voix ? Pourquoi les conseils cachés de 30 à 50 membres sont-ils hors de portée d'une énumération des coalitions ? Les réponses sont dans `05_solutions.md`, section Notebook.

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une valeur de Shapley par les ordres et par les poids des coalitions, et dire combien d'ordres tirés au hasard suffisent ?
2. Sais-tu calculer les dividendes de Harsanyi d'un jeu, et en tirer la valeur de Shapley ?
3. Sais-tu distinguer la moyenne marginale et la moyenne conditionnelle d'une fonction de variables corrélées ?
4. Sais-tu intégrer un champ le long d'une ligne brisée, reconnaître un champ de gradient, et partager une variation entre les coordonnées ?
5. Sais-tu agrandir une grille par interpolation bilinéaire, et calculer une corrélation de rangs avec des ex aequo ?
6. Sais-tu écrire les critères d'équité comme des indépendances, et dire ce qui est incompatible quand les taux de base diffèrent ?

**La suite** : le chapitre bonus B6 de workbookIA, qui applique ces outils.